# Stage 4C Notebook — Alcubierre Geometry + Discrete Proper-Time Correction Screening

This notebook is designed to run locally in **Visual Studio Code**. It includes Stage 4A, Stage 4B, and Stage 4C code for numerical curvature extraction and tensor-complete correction screening.

Install dependencies:

```bash
pip install numpy pandas matplotlib scipy ipykernel
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from scipy.optimize import least_squares
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("SciPy available:", SCIPY_AVAILABLE)

## Configuration

Start with `DIM = 3` and `N = 31`. Increase `N` after confirming that the notebook runs. Full `DIM = 4` is available but much heavier.

In [ ]:
DIM = 3               # 3 = 2+1 reduced geometry; 4 = full 3+1 geometry
N = 31                # grid points per dimension
R_BUBBLE = 3.0
SIGMA = 1.0
V_S = 0.5
EXTENT = 5.0
T_EXTENT = 0.4
DELTA_TAU = 0.04
INTERIOR_CROP = 3

## Core functions

In [ ]:
def alcubierre_shape(rs: np.ndarray, R: float, sigma: float) -> np.ndarray:
    return (np.tanh(sigma * (rs + R)) - np.tanh(sigma * (rs - R))) / (2.0 * np.tanh(sigma * R))


def make_coordinates(dim: int, N: int, extent: float, t_extent: float):
    if dim == 3:
        coords = [np.linspace(-t_extent, t_extent, N), np.linspace(-extent, extent, N), np.linspace(-extent, extent, N)]
    elif dim == 4:
        coords = [np.linspace(-t_extent, t_extent, N), np.linspace(-extent, extent, N), np.linspace(-extent, extent, N), np.linspace(-extent, extent, N)]
    else:
        raise ValueError("dim must be 3 or 4")
    spacings = [c[1] - c[0] for c in coords]
    return coords, spacings


def compute_rs(mesh, v_s: float, tau_shift: float = 0.0) -> np.ndarray:
    T, X, Y = mesh[0], mesh[1], mesh[2]
    if len(mesh) == 3:
        return np.sqrt((X - v_s * (T + tau_shift))**2 + Y**2)
    Z = mesh[3]
    return np.sqrt((X - v_s * (T + tau_shift))**2 + Y**2 + Z**2)


def finite_proper_time_seed(mesh, R, sigma, v_s, delta_tau):
    f0 = alcubierre_shape(compute_rs(mesh, v_s, 0.0), R, sigma)
    fm = alcubierre_shape(compute_rs(mesh, v_s, -delta_tau), R, sigma)
    fp = alcubierre_shape(compute_rs(mesh, v_s, +delta_tau), R, sigma)
    D2f = (fp - 2.0 * f0 + fm) / (delta_tau**2)
    S = D2f**2
    return f0, D2f, S

## Metric, curvature, and divergence functions

In [ ]:
def build_metric(f: np.ndarray, v_s: float, dim: int):
    shape = f.shape
    g = np.zeros((dim, dim) + shape)
    gi = np.zeros_like(g)
    g[0, 0] = -1.0 + v_s**2 * f**2
    g[0, 1] = -v_s * f
    g[1, 0] = -v_s * f
    g[1, 1] = 1.0
    gi[0, 0] = -1.0
    gi[0, 1] = -v_s * f
    gi[1, 0] = -v_s * f
    gi[1, 1] = 1.0 - v_s**2 * f**2
    for a in range(2, dim):
        g[a, a] = 1.0
        gi[a, a] = 1.0
    return g, gi


def metric_derivatives(g, spacings):
    dim = g.shape[0]
    shape = g.shape[2:]
    dg = np.zeros((dim, dim, dim) + shape)
    for a in range(dim):
        for b in range(dim):
            grads = np.gradient(g[a, b], *spacings, edge_order=2)
            for k in range(dim):
                dg[a, b, k] = grads[k]
    return dg


def christoffel_symbols(g, gi, spacings):
    dim = g.shape[0]
    shape = g.shape[2:]
    dg = metric_derivatives(g, spacings)
    Gamma = np.zeros((dim, dim, dim) + shape)
    for a in range(dim):
        for b in range(dim):
            for c in range(dim):
                total = np.zeros(shape)
                for l in range(dim):
                    total += gi[a, l] * (dg[c, l, b] + dg[b, l, c] - dg[b, c, l])
                Gamma[a, b, c] = 0.5 * total
    return Gamma


def ricci_and_einstein(g, gi, Gamma, spacings):
    dim = g.shape[0]
    shape = g.shape[2:]
    dGamma = np.zeros((dim, dim, dim, dim) + shape)
    for a in range(dim):
        for b in range(dim):
            for c in range(dim):
                grads = np.gradient(Gamma[a, b, c], *spacings, edge_order=2)
                for k in range(dim):
                    dGamma[a, b, c, k] = grads[k]
    Ricci = np.zeros((dim, dim) + shape)
    for b in range(dim):
        for c in range(dim):
            total = np.zeros(shape)
            for a in range(dim):
                total += dGamma[a, b, c, a] - dGamma[a, b, a, c]
                for d in range(dim):
                    total += Gamma[a, a, d] * Gamma[d, b, c]
                    total -= Gamma[a, c, d] * Gamma[d, b, a]
            Ricci[b, c] = total
    R_scalar = np.zeros(shape)
    for a in range(dim):
        for b in range(dim):
            R_scalar += gi[a, b] * Ricci[a, b]
    Einstein = np.zeros((dim, dim) + shape)
    for a in range(dim):
        for b in range(dim):
            Einstein[a, b] = Ricci[a, b] - 0.5 * g[a, b] * R_scalar
    return Ricci, R_scalar, Einstein


def mix_tensor_up_down(T_cov, gi):
    dim = T_cov.shape[0]
    Tmix = np.zeros_like(T_cov)
    for a in range(dim):
        for b in range(dim):
            for c in range(dim):
                Tmix[a, b] += gi[a, c] * T_cov[c, b]
    return Tmix


def divergence_mixed(Tmix, Gamma, spacings):
    dim = Tmix.shape[0]
    shape = Tmix.shape[2:]
    C = np.zeros((dim,) + shape)
    for b in range(dim):
        for a in range(dim):
            C[b] += np.gradient(Tmix[a, b], *spacings, edge_order=2)[a]
        for a in range(dim):
            for l in range(dim):
                C[b] += Gamma[a, a, l] * Tmix[l, b]
                C[b] -= Gamma[l, a, b] * Tmix[a, l]
    return C


def l2_norm_tensor(T, crop=3):
    dim = T.shape[0]
    if T.ndim >= 3 and T.shape[1] == dim:
        interior = tuple(slice(crop, -crop) for _ in range(T.ndim - 2))
        return np.sqrt(sum(np.mean(T[a, b][interior]**2) for a in range(dim) for b in range(dim)))
    interior = tuple(slice(crop, -crop) for _ in range(T.ndim - 1))
    return np.sqrt(sum(np.mean(T[a][interior]**2) for a in range(dim)))

## Build geometry

In [ ]:
def build_geometry(dim=DIM, N=N, R=R_BUBBLE, sigma=SIGMA, v_s=V_S, extent=EXTENT, t_extent=T_EXTENT, delta_tau=DELTA_TAU):
    coords, spacings = make_coordinates(dim, N, extent, t_extent)
    mesh = np.meshgrid(*coords, indexing="ij")
    f0, D2f, S = finite_proper_time_seed(mesh, R, sigma, v_s, delta_tau)
    g, gi = build_metric(f0, v_s, dim)
    Gamma = christoffel_symbols(g, gi, spacings)
    Ricci, R_scalar, Einstein = ricci_and_einstein(g, gi, Gamma, spacings)
    Gmix = mix_tensor_up_down(Einstein, gi)
    divG = divergence_mixed(Gmix, Gamma, spacings)
    if dim == 3:
        dfdy = np.gradient(f0, spacings[2], axis=2, edge_order=2)
        rho_A = -(v_s**2)/(32*np.pi)*dfdy**2
    else:
        dfdy = np.gradient(f0, spacings[2], axis=2, edge_order=2)
        dfdz = np.gradient(f0, spacings[3], axis=3, edge_order=2)
        rho_A = -(v_s**2)/(32*np.pi)*(dfdy**2 + dfdz**2)
    return dict(dim=dim, N=N, coords=coords, spacings=spacings, f=f0, D2f=D2f, S=S, g=g, gi=gi, Gamma=Gamma, Ricci=Ricci, R_scalar=R_scalar, Einstein=Einstein, Gmix=Gmix, divG=divG, rho_A=rho_A, sigma=sigma, v_s=v_s, delta_tau=delta_tau)

geom = build_geometry()
print('Built geometry:', geom['dim'], geom['N'], geom['f'].shape)

## Stage 4A — Bianchi identity validation

In [ ]:
G_l2 = l2_norm_tensor(geom["Gmix"], crop=INTERIOR_CROP)
divG_l2 = l2_norm_tensor(geom["divG"], crop=INTERIOR_CROP)
print("||G^a_b||_2 =", G_l2)
print("||nabla_a G^a_b||_2 =", divG_l2)
print("relative Bianchi residual =", divG_l2 / G_l2 if G_l2 > 0 else np.nan)

In [ ]:
def central_slice(field, geom):
    coords = geom['coords']
    if geom['dim'] == 3:
        t, x, y = coords
        it0 = np.argmin(np.abs(t))
        return field[it0, :, :], x, y
    t, x, y, z = coords
    it0 = np.argmin(np.abs(t))
    iz0 = np.argmin(np.abs(z))
    return field[it0, :, :, iz0], x, y


def plot_central(field, geom, title, label):
    data, x, y = central_slice(field, geom)
    plt.figure(figsize=(7, 6))
    plt.imshow(data.T, extent=[x[0], x[-1], y[0], y[-1]], origin='lower', aspect='equal')
    plt.colorbar(label=label)
    plt.title(title)
    plt.xlabel('x')
    plt.ylabel('y')
    plt.show()

plot_central(geom['rho_A'], geom, 'Known Alcubierre negative-energy density', 'rho_A')
divG_mag = np.sqrt(sum(geom['divG'][a]**2 for a in range(geom['dim'])))
plot_central(divG_mag, geom, 'Stage 4A Bianchi residual magnitude', '|∇G|')

## Stage 4B — Tensor-complete Hessian ansatz

In [ ]:
def scalar_hessian(S, geom):
    dim = geom['dim']
    spacings = geom['spacings']
    Gamma = geom['Gamma']
    gi = geom['gi']
    dS = np.zeros((dim,) + S.shape)
    grads = np.gradient(S, *spacings, edge_order=2)
    for a in range(dim): dS[a] = grads[a]
    ddS = np.zeros((dim, dim) + S.shape)
    for a in range(dim):
        grads_a = np.gradient(dS[a], *spacings, edge_order=2)
        for b in range(dim): ddS[a, b] = grads_a[b]
    Hess = np.zeros((dim, dim) + S.shape)
    for a in range(dim):
        for b in range(dim):
            Hess[a, b] = ddS[a, b]
            for l in range(dim): Hess[a, b] -= Gamma[l, a, b] * dS[l]
    BoxS = np.zeros(S.shape)
    for a in range(dim):
        for b in range(dim): BoxS += gi[a, b] * Hess[a, b]
    gradS_up = np.zeros_like(dS)
    for a in range(dim):
        for b in range(dim): gradS_up[a] += gi[a, b] * dS[b]
    return dS, gradS_up, Hess, BoxS


def make_Q_candidate(geom, candidate='H', lam=0.0, beta=0.0):
    dim = geom['dim']; g = geom['g']; S = geom['S']; Einstein = geom['Einstein']
    dS, gradS_up, Hess, BoxS = scalar_hessian(S, geom)
    Q = np.zeros((dim, dim) + S.shape)
    for a in range(dim):
        for b in range(dim):
            Q[a, b] = Hess[a, b] - g[a, b] * BoxS
            if candidate in ('HT','HTR'): Q[a, b] -= lam * g[a, b] * S
            if candidate in ('R','HTR'): Q[a, b] += beta * S * Einstein[a, b]
    return Q, {'dS': dS, 'gradS_up': gradS_up, 'Hess': Hess, 'BoxS': BoxS}

Q_H, aux_H = make_Q_candidate(geom, candidate='H')
Qmix_H = mix_tensor_up_down(Q_H, geom['gi'])
C_H = divergence_mixed(Qmix_H, geom['Gamma'], geom['spacings'])
Q_l2 = l2_norm_tensor(Qmix_H, crop=INTERIOR_CROP)
C_l2 = l2_norm_tensor(C_H, crop=INTERIOR_CROP)
print('Hessian Q ||Q||:', Q_l2)
print('Hessian Q ||div Q||:', C_l2)
print('C/Q:', C_l2/Q_l2 if Q_l2 > 0 else np.nan)

In [ ]:
plot_central(geom['S'], geom, 'Scalar finite proper-time seed S=(Dτ²f)²', 'S')
plot_central(Q_H[0,0], geom, 'Stage 4B Hessian candidate Q_00', 'Q_00')
C_H_mag = np.sqrt(sum(C_H[a]**2 for a in range(geom['dim'])))
plot_central(C_H_mag, geom, 'Stage 4B residual magnitude', '|∇Q|')

## Stage 4C — Analytic residual cancellation and candidate ranking

In [ ]:
def analytic_residual_components(geom, aux):
    dim = geom['dim']; Ricci = geom['Ricci']; Einstein = geom['Einstein']; gradS_up = aux['gradS_up']; dS = aux['dS']
    A = np.zeros_like(dS); B = dS.copy(); D = np.zeros_like(dS)
    for nu in range(dim):
        for lam in range(dim): A[nu] += Ricci[nu, lam] * gradS_up[lam]
    for nu in range(dim):
        for mu in range(dim): D[nu] += Einstein[mu, nu] * gradS_up[mu]
    return A, B, D


def crop_flat_components(fields, crop=3):
    interior = tuple(slice(crop, -crop) for _ in range(fields.ndim - 1))
    return np.concatenate([fields[a][interior].ravel() for a in range(fields.shape[0])])

A, B, D = analytic_residual_components(geom, aux_H)
a = crop_flat_components(A, INTERIOR_CROP); b = crop_flat_components(B, INTERIOR_CROP); d = crop_flat_components(D, INTERIOR_CROP)
M = np.vstack([-b, d]).T
target = -a
params, residuals, rank, svals = np.linalg.lstsq(M, target, rcond=None)
lambda_fit, beta_fit = params
res_fit = a - lambda_fit*b + beta_fit*d
print('lambda_fit =', lambda_fit)
print('beta_fit =', beta_fit)
print('fit residual ratio =', np.sqrt(np.mean(res_fit**2))/np.sqrt(np.mean(a**2)))

In [ ]:
candidate_specs = [
    ('H', 0.0, 0.0, 'H'),
    ('HT', lambda_fit, 0.0, f'HT λ={lambda_fit:.3g}'),
    ('R', 0.0, beta_fit, f'R β={beta_fit:.3g}'),
    ('HTR', lambda_fit, beta_fit, f'HTR λ={lambda_fit:.3g}, β={beta_fit:.3g}'),
    ('HTR', 0.1*lambda_fit, 0.1*beta_fit, f'HTR small λ={0.1*lambda_fit:.3g}, β={0.1*beta_fit:.3g}'),
]
rows = []
for cand, lam, beta, label in candidate_specs:
    Q, _ = make_Q_candidate(geom, candidate=cand, lam=lam, beta=beta)
    Qmix = mix_tensor_up_down(Q, geom['gi'])
    C = divergence_mixed(Qmix, geom['Gamma'], geom['spacings'])
    Q_l2 = l2_norm_tensor(Qmix, crop=INTERIOR_CROP)
    C_l2 = l2_norm_tensor(C, crop=INTERIOR_CROP)
    rows.append({'candidate': label, 'lambda': lam, 'beta': beta, 'Q_L2': Q_l2, 'C_L2': C_l2, 'C_over_Q': C_l2/Q_l2 if Q_l2 > 0 else np.nan})

df_candidates = pd.DataFrame(rows).sort_values('C_over_Q')
df_candidates

In [ ]:
plt.figure(figsize=(9,5))
plt.barh(df_candidates['candidate'], df_candidates['C_over_Q'])
plt.xscale('log')
plt.xlabel('C_L2 / Q_L2')
plt.title('Stage 4C direct candidate ranking')
plt.gca().invert_yaxis()
plt.grid(True, which='both', axis='x', alpha=0.3)
plt.show()

## Optional refinement run

In [ ]:
def run_refinement(N_values=(21, 25, 31), dim=DIM):
    rows=[]
    for n in N_values:
        print(f'Running N={n}, dim={dim}...')
        g0 = build_geometry(dim=dim, N=n)
        Q, aux = make_Q_candidate(g0, candidate='H')
        Qmix = mix_tensor_up_down(Q, g0['gi'])
        C = divergence_mixed(Qmix, g0['Gamma'], g0['spacings'])
        G_l2 = l2_norm_tensor(g0['Gmix'], crop=INTERIOR_CROP)
        divG_l2 = l2_norm_tensor(g0['divG'], crop=INTERIOR_CROP)
        Q_l2 = l2_norm_tensor(Qmix, crop=INTERIOR_CROP)
        C_l2 = l2_norm_tensor(C, crop=INTERIOR_CROP)
        A, B, D = analytic_residual_components(g0, aux)
        a = crop_flat_components(A, INTERIOR_CROP); b = crop_flat_components(B, INTERIOR_CROP); d = crop_flat_components(D, INTERIOR_CROP)
        M = np.vstack([-b, d]).T
        params, *_ = np.linalg.lstsq(M, -a, rcond=None)
        lam_fit, beta_fit = params
        res_fit = a - lam_fit*b + beta_fit*d
        rows.append({'N': n, 'dim': dim, 'relative_Bianchi_residual': divG_l2/G_l2 if G_l2 > 0 else np.nan, 'Hessian_Q_C_over_Q': C_l2/Q_l2 if Q_l2 > 0 else np.nan, 'lambda_fit': lam_fit, 'beta_fit': beta_fit, 'analytic_fit_residual_ratio': np.sqrt(np.mean(res_fit**2))/np.sqrt(np.mean(a**2)) if np.mean(a**2)>0 else np.nan})
    return pd.DataFrame(rows)

# Uncomment to run locally:
# df_refine = run_refinement(N_values=(21,25,31,37), dim=3)
# display(df_refine)

## Interpretation template

1. Stage 4A succeeds if the numerical Bianchi residual for the extracted Einstein tensor is small and decreases with refinement.
2. Stage 4B succeeds as a screening step if the constructed correction tensor is tensor-complete and wall-localized.
3. Stage 4C succeeds as a research step if the analytic residual model identifies curvature-coupled completion terms that reduce the conservation residual.
4. None of these stages proves implementability, exotic-energy cancellation, horizon stability, or chronology protection.